In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate peft wandb

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_key)

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import copy
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, pipeline
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from sklearn.metrics import confusion_matrix, classification_report
import evaluate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
try:
    df = pd.read_csv("/kaggle/input/datasets/riccardomvillaggio/tavily-data/source_quality_dataset.csv")
    print(f"Dataset caricato. Shape: {df.shape}")
except FileNotFoundError:
    raise FileNotFoundError("Dataset non trovato. Caricare il file 'source_quality_dataset.csv'.")

df["label"] = df["label"].astype(int)

dataset = Dataset.from_pandas(df)

# Split Dataset: 70% Train, 15% Validation, 15% Test
train_test = dataset.train_test_split(test_size=0.3, seed=SEED)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=SEED)

ds = DatasetDict({
    'train': train_test['train'],
    'validation': val_test['train'],
    'test': val_test['test']
})

print(ds)

In [ ]:
print(f"Dimensioni del Dataset: {len(dataset)} campioni")
print(f"  - Training Set:   {len(ds['train'])} campioni ({len(ds['train'])/len(dataset)*100:.1f}%)")
print(f"  - Validation Set: {len(ds['validation'])} campioni ({len(ds['validation'])/len(dataset)*100:.1f}%)")
print(f"  - Test Set:       {len(ds['test'])} campioni ({len(ds['test'])/len(dataset)*100:.1f}%)")

train_df = ds['train'].to_pandas()
distribuzione_train = train_df['label'].value_counts()

validation_df = ds['validation'].to_pandas()
distribuzione_validation = validation_df['label'].value_counts()

test_df = ds['test'].to_pandas()
distribuzione_test = test_df['label'].value_counts()

print("\nDistribuzione delle Classi (Training Set):")
print(f"  - Classe 1 (Articoli/Utili): {distribuzione_train.get(1, 0)} campioni")
print(f"  - Classe 0 (Junk/E-commerce): {distribuzione_train.get(0, 0)} campioni")

print("\nDistribuzione delle Classi (Validation Set):")
print(f"  - Classe 1 (Articoli/Utili): {distribuzione_validation.get(1, 0)} campioni")
print(f"  - Classe 0 (Junk/E-commerce): {distribuzione_validation.get(0, 0)} campioni")

print("\nDistribuzione delle Classi (Test Set):")
print(f"  - Classe 1 (Articoli/Utili): {distribuzione_test.get(1, 0)} campioni")
print(f"  - Classe 0 (Junk/E-commerce): {distribuzione_test.get(0, 0)} campioni")

print("ESEMPIO DI CAMPIONE DAL TRAINING SET:")
campione_casuale = ds['train'][0]
label_name = "Articolo/Utile (1)" if campione_casuale['label'] == 1 else "Junk/E-commerce (0)"

print(f"LABEL: {label_name}")
print(f"TESTO:\n{campione_casuale['testo'][:400]}...")

In [ ]:
model_checkpoint = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(example):
    return tokenizer(example["testo"], truncation=True)

tokenized_datasets = ds.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["testo"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

print("Colonne presenti nel dataset:", tokenized_datasets["train"].column_names)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    acc = metric_acc.compute(predictions=preds, references=labels)
    f1 = metric_f1.compute(predictions=preds, references=labels, average="macro")
    
    return {
        "accuracy": acc["accuracy"],
        "f1_macro": f1["f1"]
    }

In [ ]:
NUM_LABELS = 2

hyperparameter_grid = [
    {"lr": 5e-5, "lora_r": 8, "epochs": 5},
    {"lr": 2e-4, "lora_r": 2, "epochs": 5},
    {"lr": 2e-4, "lora_r": 4, "epochs": 5},
    {"lr": 2e-4, "lora_r": 8, "epochs": 5},
    {"lr": 2e-4, "lora_r": 16, "epochs": 5},
    {"lr": 2e-4, "lora_r": 32, "epochs": 5},
    {"lr": 2e-4, "lora_r": 64, "epochs": 5}
]

best_f1 = 0
best_config = None
best_model_dir = "./best_lora_model"

results_log = []

os.environ["WANDB_PROJECT"] = "CCAI_Source_Evaluator"

print("Inizio Hyperparameter Search...")

for idx, config in enumerate(hyperparameter_grid):
    print(f"\n--- Testing Config {idx+1}/{len(hyperparameter_grid)}: {config} ---")
    
    temp_base_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=NUM_LABELS)
    temp_lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=config["lora_r"], lora_alpha=32, 
        lora_dropout=0.1, bias="none", target_modules=["query", "value"]
    )
    temp_peft_model = get_peft_model(temp_base_model, temp_lora_config)
    temp_peft_model.print_trainable_parameters()
    temp_peft_model.cuda()
    
    temp_training_args = TrainingArguments(
        output_dir=f"./lora_search_{idx}",
        learning_rate=config["lr"],
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        num_train_epochs=config["epochs"],
        weight_decay=0.01,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        report_to="wandb",
        run_name=f"LoRA_lr-{config['lr']}_r-{config['lora_r']}",
        fp16=True,
        seed=42
    )
    
    temp_trainer = Trainer(
        model=temp_peft_model,
        args=temp_training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        data_collator=data_collator
    )
    
    temp_trainer.train()
    val_results = temp_trainer.evaluate()
    val_f1 = val_results["eval_f1_macro"]
    
    results_log.append({
        "config": config,
        "val_f1": val_f1,
        "val_acc": val_results["eval_accuracy"]
    })
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_config = config
        temp_trainer.save_model(best_model_dir)
        
    wandb.finish()

print("\n--- RISULTATI HYPERPARAMETER SEARCH ---")
for res in results_log:
    print(f"Config: {res['config']} -> Val F1: {res['val_f1']:.4f} | Val Acc: {res['val_acc']:.4f}")
print(f"\nMiglior Configurazione scelta: {best_config} con F1={best_f1:.4f}")

In [ ]:
print("Caricamento del miglior modello per il test...")
base_model_for_eval = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=NUM_LABELS)
best_peft_model = PeftModel.from_pretrained(base_model_for_eval, best_model_dir)

final_trainer = Trainer(
    model=best_peft_model,
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

finetuned_val = final_trainer.evaluate()

def baseline_heuristic(text: str) -> int:
    text_lower = text.lower()
    junk_keywords = [
    "acquista", "prezzo", "carrello", "ebay", "spedizione", "facebook", "accedi", 
    "cookie", "sconto", "pre-order", "password", "newsletter", "abbonati", 
    "privacy", "aggiungi", "store", "offerta", "login", "registrati"
    ]
    if any(kw in text_lower for kw in junk_keywords) or len(text_lower) < 30:
        return 0 
    return 1

true_labels = ds["test"]["label"]
baseline_preds = [baseline_heuristic(text) for text in ds["test"]["testo"]]

baseline_acc = metric_acc.compute(predictions=baseline_preds, references=true_labels)["accuracy"]
baseline_f1 = metric_f1.compute(predictions=baseline_preds, references=true_labels, average="macro")["f1"]

print("Valutazione Zero-Shot Baseline")
zero_shot_pipe = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=0)

zero_shot_preds = []
candidate_labels = ["informative article, review or analysis", "online store, product page, login page, forum or community page"]

for text in ds["test"]["testo"]:
    result = zero_shot_pipe(text, candidate_labels)

    if result['labels'][0] == "informative article, review or analysis":
        zero_shot_preds.append(1)
    else:
        zero_shot_preds.append(0)

zs_acc = metric_acc.compute(predictions=zero_shot_preds, references=true_labels)["accuracy"]
zs_f1 = metric_f1.compute(predictions=zero_shot_preds, references=true_labels, average="macro")["f1"]

print("\n" + "="*50)
print("COMPARAZIONE FINALE (TEST SET) - I TRE LIVELLI")
print("="*50)

print(f"Livello 1 - Baseline (Heuristic Rule):")
print(f"  Accuracy: {baseline_acc:.4f}  |  F1 Macro: {baseline_f1:.4f}")

print(f"\nLivello 2 - Baseline (Zero-Shot AI - mDeBERTa-v3):")
print(f"  Accuracy: {zs_acc:.4f}  |  F1 Macro: {zs_f1:.4f}")

print(f"\nLivello 3 - Modello Fine-Tuned (LoRA):")
print(f"  Accuracy: {finetuned_val['eval_accuracy']:.4f}  |  F1 Macro: {finetuned_val['eval_f1_macro']:.4f}")

miglioramento = finetuned_val['eval_accuracy'] - baseline_acc
print(f"\nMiglioramento Assoluto LoRA vs Heuristic: {miglioramento*100:+.2f}%")

print("\n" + "="*50)
print("ERROR ANALYSIS (Sul Modello LoRA)")
print("="*50)

preds_output = final_trainer.predict(tokenized_datasets["test"])
preds = np.argmax(preds_output.predictions, axis=-1)
refs = preds_output.label_ids

label_names = ["Junk/Store (0)", "Informative (1)"]

print("\nClassification Report:")
print(classification_report(refs, preds, target_names=label_names, digits=3))

print("Confusion Matrix:")
print(confusion_matrix(refs, preds))

wrong_idx = np.where(preds != refs)[0]
print("\nNumber of errors:", len(wrong_idx))

test_texts = ds["test"]["testo"]

for i in wrong_idx[:5]: # Mostriamo i primi 5 errori
    text = test_texts[int(i)]
    vera_label = label_names[int(refs[i])]
    pred_label = label_names[int(preds[i])]
    
    print("-" * 80)
    print(f"TRUE={vera_label} | PRED={pred_label}")
    print(text[:350], "...")